# Run the GramVet livestock disease classifier

Run these cells from top to bottom. The training benchmark includes 5-fold cross-validation and can take several minutes.

> **Important:** The dataset is synthetic. Metrics and calibrated probabilities only describe this simulator and do not establish clinical accuracy on real animals. See `clinical_sources.md` and `README.md`.

In [ ]:
from pathlib import Path
import subprocess, sys

# Find the extracted GramVet_ML package even if Jupyter starts in a child folder.
candidates = [Path.cwd(), *Path.cwd().parents]
PACKAGE_DIR = next((p for p in candidates if (p / 'generate_dataset.py').exists() and (p / 'models' / 'train_disease_model.py').exists()), None)
if PACKAGE_DIR is None:
    raise FileNotFoundError('Open this notebook from the extracted GramVet_ML folder.')
print('Package folder:', PACKAGE_DIR)
print('Python:', sys.version.split()[0])

## 1. Install required packages

Run this once in the notebook environment. Internet access may be needed to install packages.

In [ ]:
%pip install -r {PACKAGE_DIR / 'requirements.txt'}

## 2. Generate the synthetic dataset

The default creates 12,600 rows with the fixed 32-feature schema and seven requested labels. This is a balanced development sample, not real case data or a prevalence estimate.

In [ ]:
subprocess.run([sys.executable, str(PACKAGE_DIR / 'generate_dataset.py'), '--rows-per-class', '1800', '--seed', '20260926'], cwd=PACKAGE_DIR, check=True)

## 3. Train and evaluate

This benchmarks four tabular models with stratified 5-fold cross-validation, tunes the top candidates using training data, calibrates with the validation split, then evaluates the held-out test split. The final test set is not used to select or calibrate the model.

In [ ]:
subprocess.run([sys.executable, str(PACKAGE_DIR / 'models' / 'train_disease_model.py')], cwd=PACKAGE_DIR, check=True)

## 4. Review the evaluation and data-quality reports

In [ ]:
import json, pandas as pd
report = json.loads((PACKAGE_DIR / 'reports' / 'evaluation_report.json').read_text(encoding='utf-8'))
quality = json.loads((PACKAGE_DIR / 'reports' / 'dataset_quality.json').read_text(encoding='utf-8'))
print('Selected model:', report['selected_model'])
print('Test metrics:')
display(pd.Series(report['test_final'], name='test'))
print('Data checks:', {k: quality[k] for k in ['rows', 'feature_count', 'missing_cells', 'exact_duplicate_rows', 'duplicate_input_rows', 'near_duplicate_screen']})
print('Full report:', PACKAGE_DIR / 'reports' / 'evaluation_report.md')

## 5. Try disease prediction

This example takes one 32-feature row from the generated CSV. Replace `sample_input` with a dictionary containing all required fields to try another case. Inference ignores extra fields but requires every declared feature.

In [ ]:
sys.path.insert(0, str(PACKAGE_DIR))
from predict_disease import predict_disease
dataset = pd.read_csv(PACKAGE_DIR / 'data' / 'livestock_clinical_synthetic.csv')
sample_input = dataset.iloc[0].drop(labels='Disease').to_dict()
prediction = predict_disease(sample_input)
prediction


In [ ]:
# Probabilities are returned as percentages and rounded to one decimal place.
print('Sum of displayed probabilities:', sum(prediction['probabilities'].values()))
print('Confidence equals predicted class probability:', prediction['confidence'] == prediction['probabilities'][prediction['predicted_disease']])

## Files created

- `data/livestock_clinical_synthetic.csv` — generated training data
- `models/disease_classifier.joblib` — preprocessing pipeline, classifier, and temperature calibration
- `reports/evaluation_report.md` and `.json` — model metrics, calibration, and confusion matrix
- `reports/dataset_quality.json` — schema, distributions, duplicates, and leakage checks

For terminal use instead of the notebook, see `README.md`.